In [ ]:
import logging
import os
from datetime import datetime

import numpy as np
import pandas as pd
from src.functions import RenamerExifTool
from tqdm.auto import tqdm

In [ ]:
logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s [%(levelname)s] %(message)s"
)
# pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Parameters

In [ ]:
DIR_SC = "sc"  # Source
DIR_ST = "st"  # Staging
DIR_FT = "ft"  # Fact


current_time = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
PATH_CSV = f"mapping_{current_time}.csv"  # Save mapping of filenames before -> after
COLS = [
    "SourceFile",
    "File:Directory",
    "File:FileName",
    "File:FileTypeExtension",
    "File:FileSize",
    "QuickTime:Model",
    "QuickTime:CreateDate",
    "QuickTime:GoPro_CPIN",
    "EXIF:Make",
    "EXIF:Model",
    "EXIF:CreateDate",
    "Composite:SubSecCreateDate",
    "MakerNotes:ShutterCount",
    "tag_date",
    "warning_msg",
    "info_msg",
    "datetime_capture_sec",
    "filename_new",
]

# Create df without actual renaming

In [ ]:
renamer = RenamerExifTool(
    dir_sc=DIR_SC,
    dir_st=DIR_ST,
    dir_ft=DIR_FT,
    path_csv=PATH_CSV,
)
df = renamer.run(output=False)
print(df.shape)

In [ ]:
cols = [x for x in COLS if x in df]
df[cols].head()

# Count warnings

In [ ]:
df["warning_msg"] = df["warning_msg"].fillna("")
if df[df["warning_msg"] != ""].shape[0] == 0:
    print("No warnings")
if df[df["warning_msg"] != ""].shape[0] > 0:
    df[df["warning_msg"] != ""][cols].head()

# Unique info

In [ ]:
quicktime = df.get("QuickTime:Model", pd.Series(np.nan, index=df.index))
exif = df.get("EXIF:Model", pd.Series(np.nan, index=df.index))

df["manufacturer"] = np.where(
    quicktime.notnull() & (quicktime != ""),
    quicktime,
    np.where(exif.notnull() & (exif != ""), exif, "Unknown"),
)

df.groupby("manufacturer")["info_msg"].value_counts(dropna=False)

# Rename

In [ ]:
assert df[df["warning_msg"] != ""].shape[0] == 0
renamer.run(output=True)